## <span style="color: Violet;"> **Импорт библиотек и Интерфейсы** </style> 

In [1]:
import json
import os
import copy
from abc import ABC, abstractmethod
from typing import List, Optional, Dict, Any

class IUnit(ABC):
    @property
    @abstractmethod
    def name(self) -> str: pass
    
    @property
    @abstractmethod
    def hp(self) -> int: pass
    
    @abstractmethod
    def take_damage(self, amount: int) -> None: pass

class ILandscape(ABC):
    @property
    @abstractmethod
    def symbol(self) -> str: pass

    @property
    @abstractmethod
    def move_cost(self) -> int: pass

class INeutralObject(ABC):
    @property
    @abstractmethod
    def symbol(self) -> str: pass

    @abstractmethod
    def effect(self, unit: 'IUnit') -> bool:
        """Возвращает True, если объект поглощается/уничтожается после взаимодействия"""
        pass


## <span style="color: Plum;"> **Окружение и Нейтральные объекты** </style> 

In [2]:
# Landscapes

class Plain(ILandscape):
    symbol = '.'
    move_cost = 1

class Forest(ILandscape):
    symbol = '🌲'
    move_cost = 2

class Mountain(ILandscape):
    symbol = "🗻"
    move_cost = 99

class Reaver(ILandscape):
    symbol = '_'
    move_cost = 4

# Neutral Objects

class HealingShrine(INeutralObject):
    symbol = '⛩️'
    def effect(self, unit: IUnit) -> bool:
        hp_before = unit.hp
        unit.hp = min(unit.hp + 20, unit.max_hp)
        hp_restored = unit.hp - hp_before
        print(f"Древнее божество услышало молитвы {unit.name} ")
        print(f"{unit.name} восстановил {hp_restored} HP")

class Rill(INeutralObject):
    symbol = '='
    def effect(self, unit: IUnit) -> bool:
        hp_before = unit.hp
        unit.hp = min(unit.hp + 5, unit.max_hp)
        hp_restored = unit.hp - hp_before
        print(f"{unit.name} выпил водички из родника")
        print(f"{unit.name} восстановил {hp_restored} HP")

class Chest(INeutralObject):
    symbol = '❌'
    def effect(self, unit: IUnit) -> bool:
        print(f"{unit.name} обнаружил сундук, однако удача отвернулась от него: внутри было пусто")
        return True

class Trap(INeutralObject):
    symbol = '_'
    def effect(self, unit: IUnit) -> bool:
        unit.take_damage(20)
        print(f"{unit.name} угодил в ловушку")
        print(f"{unit.name} потерял 20 HP")
        return True

class EnrichedMine(INeutralObject):
    symbol = '⛏️'
    def effect(self, unit: IUnit) -> bool:
        print(f"{unit.name} успешно извлёк ценные ресурсы из недр обогащённого рудника")
        return False


## <span style="color: purple;"> **Юниты** </style> 

In [ ]:
class Unit(IUnit):
    def __init__(self, name: str, hp: int, attack: int, armor: int, move_range: int):
        self._name = name
        self.max_hp = hp
        self._hp = hp
        self.attack = attack
        self.armor = armor
        self.move_range = move_range
        self.x = 0
        self.y = 0

        @property
        def name(self) -> str: return self._name

        @property
        def hp(self) -> int: return self._hp

        @hp.setter
        def hp(self, value: int): self._hp = max(0, value)

        def take_damage(self, amount: int) -> None:
            damage = max(0, amount - self.armor)
            self.hp -= damage

        def __add__(self, other: INeutralObject):
            if isinstance(other, INeutralObject):
                is_consumed = other.effect(self)
                return is_consumed
            raise TypeError("Юнит может взаимодействовать с INeutralObject только через оператор «+»")
# Единицы ближнего боя
class Zealot(Unit):
    def __init__(self): super().__init__("Зилот", hp=100, attack = 15, armor = 15, move_range = 3)
class DarkTemplar(Unit):
    def __init__(self): super().__init__("Тёмный тамплиер", hp=50, attack = 30, armor = 5, move_range = 4)
# Единицы дальнего боя
class Dragoon(Unit):
    def __init__(self): super().__init__("Драгун", hp=120, attack = 20, armor = 20, move_range = 3)
class Stalker(Unit):
    def __init__(self): super().__init__("Сталкер", hp=100, attack = 25, armor = 10, move_range = 4)
# Тяжёлые механические единицы
class Reaver(Unit):
    def __init__(self): super().__init__("Опустошитель", hp=100, attack = 80, armor = 5, move_range = 1)
class Immortal(Unit):
    def __init__(self): super().__init__("Опустошитель", hp=150, attack = 20, armor = 40, move_range = 2)
class Colossus(Unit):
    def __init__(self): super().__init__("Опустошитель", hp=200, attack = 40, armor = 20, move_range = 8)


## <span style="color: Lavender;"> **Задача 19(Работа с JSON)** </style> 

In [4]:
import os
import json
from typing import Any
class JsonManager:
    "Инициализация файла"
    def __init__ (self, filepath, str):
        self.filepath = filepath
        if not os.path.exists(self.filepath):
            self.write()

    "Функция для чтения файла"
    def read(self) -> dict:
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            return {}
            
    "Функция для полной перезаписи файла"
    def write(self, data: dict) -> None:
        if not isinstance(data, dict):
            raise TypeError("Корень JSON должен быть словарём (dict) для корректной работы методов.")
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

    "Функция для добавление новой пары ключ-значение"
    def add(self, key: str, value: Any) -> None:
        data = self.read()
        if key in data:
            raise ValueError(f"Ключ {key} уже существует, используйте update().")
        data[key]=value
        self.write(data)

    "Функция для обновления ключа" 
    def update(self, key: str, value: Any) -> None:
        data = self.read()
        if key not in data:
            raise KeyError(f"Ключ {key} не найден")
        data[key] = value
        self.write(data)

    "Функция для удаления ключа"
    def remove(self, key: str, value: Any) -> None:
            data = self.read()
            if key not in data:
                raise KeyError(f"Ключ {key} не найден")
            del data[key]
            self.write(data)
    "Функция для получения ключа"
    def get(self, key: str, default=None):
        return self.read().get(key, default)